# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

# Global 18-way fingerprint model plus version-family specialist models.
texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

specialist_groups = [
    ['gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash'],
    ['gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra'],
    ['glm-5.3', 'glm-5.3-flash'],
    ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5'],
]

def style_one(text):
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words, n_chars = max(len(words),1), max(len(text),1)
    sentences = max(len(re.findall(r'[.!?]+', text)),1)
    lengths = np.asarray([len(w) for w in words], dtype=float)
    punct = '.,;:!?()[]{}"\'/-'
    return np.asarray([
        np.log1p(len(text)), np.log1p(len(words)), np.log1p(sentences),
        lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
        len(set(words))/n_words, sum(c.isdigit() for c in text)/n_chars,
        sum(c.isupper() for c in text)/n_chars, sum(c.isspace() for c in text)/n_chars,
        text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars,
        *[text.count(c)/n_chars for c in punct], text.count('#')/n_chars,
        text.count('*')/n_chars, text.count('```')/n_chars,
    ], dtype=np.float32)

def make_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1,3), min_df=2, sublinear_tf=True, max_features=120000)),
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,6), min_df=2, sublinear_tf=True, max_features=150000)),
    ])

style = np.vstack([style_one(x) for x in texts])
all_global, all_refined = [], []
fold_results = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(texts, labels, groups=groups), 1):
    tf = make_features()
    Xtr_text = tf.fit_transform(texts[tr])
    Xva_text = tf.transform(texts[va])
    sc = StandardScaler()
    Xtr_style = sc.fit_transform(style[tr])
    Xva_style = sc.transform(style[va])
    Xtr = hstack([Xtr_text, csr_matrix(Xtr_style)], format='csr')
    Xva = hstack([Xva_text, csr_matrix(Xva_style)], format='csr')
    ytr, yva = labels[tr], labels[va]

    global_model = LinearSVC(C=1.0, class_weight='balanced', max_iter=1200)
    global_model.fit(Xtr, ytr)
    pred_global = global_model.predict(Xva)
    pred_refined = pred_global.copy()

    specialist_scores = {}
    for group in specialist_groups:
        train_mask = np.isin(ytr, group)
        val_mask = np.isin(pred_global, group)
        if not np.any(val_mask):
            continue
        local = LinearSVC(C=1.0, class_weight='balanced', max_iter=1200)
        local.fit(Xtr[train_mask], ytr[train_mask])
        pred_refined[val_mask] = local.predict(Xva[val_mask])
        specialist_scores[' / '.join(group)] = int(val_mask.sum())

    global_ba = balanced_accuracy_score(yva, pred_global)
    refined_ba = balanced_accuracy_score(yva, pred_refined)
    fold_results.append({'fold': fold, 'global_balanced_accuracy': float(global_ba), 'specialist_balanced_accuracy': float(refined_ba), 'rows_refined': specialist_scores})
    all_global.extend(zip(yva.tolist(), pred_global.tolist()))
    all_refined.extend(zip(yva.tolist(), pred_refined.tolist()))
    print(f"Fold {fold}: global={global_ba:.4f}, specialist-refined={refined_ba:.4f}, rows={specialist_scores}")

summary = {
    'experiment': 'global 18-way plus version-specialist refinement',
    'split_method': '5-fold GroupKFold by prompt_id',
    'fold_results': fold_results,
    'global_mean': float(np.mean([r['global_balanced_accuracy'] for r in fold_results])),
    'global_std': float(np.std([r['global_balanced_accuracy'] for r in fold_results], ddof=1)),
    'specialist_mean': float(np.mean([r['specialist_balanced_accuracy'] for r in fold_results])),
    'specialist_std': float(np.std([r['specialist_balanced_accuracy'] for r in fold_results], ddof=1)),
    'global_report': classification_report([x[0] for x in all_global], [x[1] for x in all_global], output_dict=True, zero_division=0),
    'specialist_report': classification_report([x[0] for x in all_refined], [x[1] for x in all_refined], output_dict=True, zero_division=0),
}
print('\\nSUMMARY:', json.dumps(summary, indent=2))
Path('/kaggle/working/specialist_results.json').write_text(json.dumps(summary, indent=2))
print('Saved:', '/kaggle/working/specialist_results.json')


## Global plus version-specialist diagnostic

This run compares a direct 18-way model with controlled specialist refinement within difficult model-version groups.
